In [0]:
# ============================================================
# Delta Lake Advanced Engineering & Optimization
# Databricks + Delta Lake
# ============================================================

# ============================================================
# 1. Importações
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
import random

# ============================================================
# 2. Criação da massa de dados
# ============================================================

linhas = 50_000_000
df = (
    spark.range(linhas)
    .withColumnRenamed("id","id_venda")
    .withColumn(
        "data_venda",
        date_add(
            lit("2024-01-01"),
            (rand() * 730).cast("int")
        )
    )
    .withColumn(
        "id_cliente",
        (rand() * 100).cast("int")
    )
    .withColumn(
        "id_produto",
        (rand() * 50).cast("int")
    )
    .withColumn(
        "id_loja",
        (rand() * 100).cast("int")
    )
    .withColumn(
        "quantidade",
        (rand() * 10 + 1).cast("int")
    )
    .withColumn(
        "valor_unitario",
        round(rand()*100,2)
    )
    .withColumn(
        "valor_total",
        (col("quantidade") * col("valor_unitario")).cast(DecimalType(15,2))
    )
)

In [0]:
%%sql
create schema IF NOT EXISTS performance

tabela = "performance.fato_vendas"

spark.sql(f"DROP TABLE IF EXISTS {tabela}")

df.repartition(200) \
.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(tabela)

In [0]:
%sql
DESCRIBE detail performance.fato_vendas;

Benchmark Inicial

Cenário 1

In [0]:
%sql
SELECT
    COUNT(*) AS total_vendas,
    SUM(valor_total) AS faturamento
FROM performance.fato_vendas
WHERE data_venda BETWEEN '2025-01-01' AND '2025-03-31';


Cenário 2

In [0]:
%sql
SELECT
    *
FROM performance.fato_vendas
WHERE id_cliente = 79;

Cenário 3

In [0]:
%sql
SELECT
    id_loja,
    SUM(valor_total) faturamento
FROM performance.fato_vendas
GROUP BY id_loja;

Aplicar OPTIMIZE

In [0]:
%sql
OPTIMIZE performance.fato_vendas
ZORDER BY (id_cliente);

Validara redução de numFiles

In [0]:
%sql
DESCRIBE DETAIL performance.fato_vendas;

Benchmark após OPTIMIZE

Cenário 1

In [0]:
%sql
SELECT
    COUNT(*) AS total_vendas,
    SUM(valor_total) AS faturamento
FROM performance.fato_vendas
WHERE data_venda BETWEEN '2025-01-01'
AND '2025-03-31';

Cenário 2

In [0]:
%sql
SELECT
    *
FROM performance.fato_vendas
WHERE id_cliente = 79;

Cenário 3

In [0]:
%sql
SELECT
    id_loja,
    SUM(valor_total) AS faturamento
FROM performance.fato_vendas
GROUP BY id_loja;

Aplicar ZORDER

Como a principal consulta por filtro é cliente:

In [0]:
%sql
OPTIMIZE performance.fato_vendas
ZORDER BY (id_cliente);

In [0]:
%sql
SELECT *
FROM performance.fato_vendas
WHERE id_cliente = 79;

Time Travel

Ver histórico:

In [0]:
%sql
DESCRIBE HISTORY performance.fato_vendas;

In [0]:
%sql
UPDATE performance.fato_vendas
SET valor_total = 0
WHERE id_cliente = 79;

Validar:

In [0]:
%sql
SELECT *
FROM performance.fato_vendas
where id_venda = 771

Consultar versão anterior

In [0]:
%sql
SELECT *
FROM performance.fato_vendas
VERSION AS OF 1
WHERE id_venda = 771;

Restaurar tabela

In [0]:
%sql
RESTORE TABLE performance.fato_vendas
TO VERSION AS OF 1;

Validar

In [0]:
%sql
SELECT *
FROM performance.fato_vendas
where id_venda = 771